# FinTrust Digital Bank — Week 3 Advanced Data Analytics

Extends Week 2 into deeper business intelligence and validation. Only 1 of 33 transaction Customer_IDs matches the visible customer dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
customers=pd.read_csv("../data/FinTrust_Customer_Data_Week2.csv")
transactions=pd.read_csv("../data/FinTrust_Transaction_Data_Week2.csv",parse_dates=["Transaction_DateTime"])
transactions["Hour"]=transactions["Transaction_DateTime"].dt.hour
transactions["Amount_Band"]=pd.cut(transactions["Amount_NGN"],[-np.inf,1000,5000,10000,50000,np.inf],labels=["<₦1k","₦1k–₦5k","₦5k–₦10k","₦10k–₦50k",">₦50k"])
print("Customers:",len(customers),"Transactions:",len(transactions),"Total value:",round(transactions["Amount_NGN"].sum(),2))

## 1. Relationship integrity

In [ ]:
transactions["Customer_ID_Match"]=transactions["Customer_ID"].isin(set(customers["Customer_ID"]))
transactions["Customer_ID_Match"].value_counts()

Result: 1/33 transaction IDs match. Customer-level transaction segmentation is withheld.

## 2. Channel performance

In [ ]:
channel=transactions.groupby("Channel").agg(Transactions=("Transaction_ID","count"),Total_Value_NGN=("Amount_NGN","sum"),Average_Value_NGN=("Amount_NGN","mean")).reset_index()
channel["Value_Share_Pct"]=channel["Total_Value_NGN"]/transactions["Amount_NGN"].sum()*100
channel.sort_values("Total_Value_NGN",ascending=False).round(2)

In [ ]:
plt.figure(figsize=(9,5)); plt.bar(channel["Channel"],channel["Transactions"]); plt.title("Transaction Volume by Channel"); plt.ylabel("Transactions"); plt.xticks(rotation=25); plt.tight_layout(); plt.show()

Validation: Mobile App remains first by count and value — 14 transactions and ₦493,341.73 (51.4% of value).

## 3. Transaction type

In [ ]:
types=transactions.groupby("Transaction_Type").agg(Transactions=("Transaction_ID","count"),Total_Value_NGN=("Amount_NGN","sum"),Average_Value_NGN=("Amount_NGN","mean")).reset_index()
types["Value_Share_Pct"]=types["Total_Value_NGN"]/transactions["Amount_NGN"].sum()*100
types.sort_values("Total_Value_NGN",ascending=False).round(2)

In [ ]:
plt.figure(figsize=(9,5)); plt.bar(types["Transaction_Type"],types["Total_Value_NGN"]); plt.title("Transaction Value by Type"); plt.ylabel("Total Value (NGN)"); plt.xticks(rotation=25); plt.tight_layout(); plt.show()

Validation: Cash Withdrawal remains first by value at ₦395,660.05 (41.2%).

## 4. Outliers and concentration

In [ ]:
q1=transactions["Amount_NGN"].quantile(.25); q3=transactions["Amount_NGN"].quantile(.75); upper=q3+1.5*(q3-q1)
outliers=transactions[transactions["Amount_NGN"]>upper].sort_values("Amount_NGN",ascending=False)
print("Upper threshold:",round(upper,2),"Outliers:",len(outliers),"Value share:",round(outliers["Amount_NGN"].sum()/transactions["Amount_NGN"].sum()*100,1),"%")
outliers[["Transaction_ID","Transaction_Type","Channel","Amount_NGN"]]

In [ ]:
ranked=transactions[["Transaction_ID","Amount_NGN"]].sort_values("Amount_NGN",ascending=False).reset_index(drop=True)
ranked["Rank"]=ranked.index+1; ranked["Cumulative_Value_Share_Pct"]=ranked["Amount_NGN"].cumsum()/transactions["Amount_NGN"].sum()*100
plt.figure(figsize=(9,5)); plt.plot(ranked["Rank"],ranked["Cumulative_Value_Share_Pct"],marker="o"); plt.axhline(80,linestyle="--"); plt.title("Cumulative Transaction Value Concentration"); plt.xlabel("Transactions ranked by amount"); plt.ylabel("Cumulative value share (%)"); plt.tight_layout(); plt.show()

Validation: Five IQR outliers account for 71.4% of total value.

## 5. Amount bands

In [ ]:
bands=transactions.groupby("Amount_Band",observed=False).agg(Transactions=("Transaction_ID","count"),Total_Value_NGN=("Amount_NGN","sum"),Average_Value_NGN=("Amount_NGN","mean")).reset_index()
bands["Value_Share_Pct"]=bands["Total_Value_NGN"]/transactions["Amount_NGN"].sum()*100
bands.round(2)

In [ ]:
plt.figure(figsize=(9,5)); plt.bar(bands["Amount_Band"].astype(str),bands["Total_Value_NGN"]); plt.title("Transaction Value by Amount Band"); plt.ylabel("Total Value (NGN)"); plt.xticks(rotation=20); plt.tight_layout(); plt.show()

## 6. Hourly analysis

In [ ]:
hourly=transactions.groupby("Hour").agg(Transactions=("Transaction_ID","count"),Total_Value_NGN=("Amount_NGN","sum"),Average_Value_NGN=("Amount_NGN","mean")).reset_index()
hourly.round(2)

In [ ]:
plt.figure(figsize=(9,5)); plt.plot(hourly["Hour"],hourly["Total_Value_NGN"],marker="o"); plt.title("Transaction Value by Hour"); plt.xlabel("Hour"); plt.ylabel("Total Value (NGN)"); plt.xticks(hourly["Hour"]); plt.tight_layout(); plt.show()

Limitation: all visible transactions occur on 2026-01-01 from 00:00 to 05:45, so hourly results are exploratory rather than full-day operational trends.

## Validation summary
- Mobile App leads volume/value — validated
- Everyday is largest visible segment — validated
- Mean exceeds median/right skew — validated
- Cash Withdrawal leads value — validated
- Customer_ID relationship is unreliable — validated

Transaction_Status and Risk_Review_Flag are absent and are not fabricated.